# Logistic Map: Interactive Dynamics

This notebook explores how the behavior of the logistic map changes as we vary the parameter $r$.

$$x_{t+1}=r x_t(1-x_t)$$

Use the **r slider** to change the parameter, and the **Play / Iteration controls** to move through the dynamics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output


In [ ]:
# Logistic rule
def logistic(x, r):
    return r * x * (1 - x)


# Calculate one full trajectory
def calculate_dynamics(r, x0=0.1, n_iter=50):

    px, py = [x0], [0]
    tx, ty = [0], [x0]

    x = x0

    for t in range(1, n_iter + 1):

        x_next = logistic(x, r)

        # Cobweb: vertical then horizontal
        px.extend([x, x_next])
        py.extend([x_next, x_next])

        # Time series
        tx.append(t)
        ty.append(x_next)

        x = x_next

    return px, py, tx, ty


In [ ]:
# Controls
r_slider = widgets.FloatSlider(
    value=2.5,
    min=2.5,
    max=4.0,
    step=0.01,
    description='r:',
    continuous_update=False,
    readout_format='.2f'
)

iteration_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=50,
    step=1,
    description='Iteration:'
)

play = widgets.Play(
    value=0,
    min=0,
    max=50,
    step=1,
    interval=150
)

widgets.jslink(
    (play, 'value'),
    (iteration_slider, 'value')
)

output = widgets.Output()


In [ ]:
def redraw(*args):

    r = r_slider.value
    frame = iteration_slider.value

    px, py, tx, ty = calculate_dynamics(
        r=r,
        x0=0.1,
        n_iter=50
    )

    with output:

        clear_output(wait=True)

        fig, (ax1, ax2) = plt.subplots(
            1, 2,
            figsize=(12, 5)
        )

        xs = np.linspace(0, 1, 300)

        # Cobweb
        ax1.plot(
            xs,
            logistic(xs, r),
            'k',
            lw=1.2
        )

        ax1.plot(
            xs,
            xs,
            'k--',
            alpha=0.4
        )

        cob_idx = min(
            1 + frame * 2,
            len(px)
        )

        ax1.plot(
            px[:cob_idx],
            py[:cob_idx],
            'r-',
            lw=1
        )

        if cob_idx > 0:
            ax1.plot(
                px[cob_idx - 1],
                py[cob_idx - 1],
                'ro',
                markersize=5
            )

        ax1.set_xlim(0, 1)
        ax1.set_ylim(0, 1)
        ax1.set_xlabel('$x_t$')
        ax1.set_ylabel('$x_{t+1}$')
        ax1.set_title(f'Cobweb — r = {r:.2f}')

        # Time series
        ax2.plot(
            tx[:frame + 1],
            ty[:frame + 1],
            'b-o',
            lw=1.2,
            markersize=4
        )

        ax2.set_xlim(0, 50)
        ax2.set_ylim(0, 1)
        ax2.set_xlabel('Iteration')
        ax2.set_ylabel('$x_t$')
        ax2.set_title('Time Series')
        ax2.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()


In [ ]:
# Give widgets enough room in Colab
r_slider.layout = widgets.Layout(width='350px')
play.layout = widgets.Layout(width='120px')
iteration_slider.layout = widgets.Layout(width='350px')

r_slider.observe(redraw, names='value')
iteration_slider.observe(redraw, names='value')

controls = widgets.VBox([
    widgets.HBox([
        widgets.Label('Parameter:'),
        r_slider
    ]),
    widgets.HBox([
        widgets.Label('Dynamics:'),
        play,
        iteration_slider
    ])
])

display(controls, output)
redraw()
